In [2]:
from groundingdino.util.inference import load_model, load_image, predict, annotate
import cv2

model = load_model(r"D:\3d-recon\Grounded-Segment-Anything\GroundingDINO\groundingdino\config\GroundingDINO_SwinT_OGC.py", r"D:\3d-recon\Grounded-Segment-Anything\groundingdino_swint_ogc.pth", device="cuda"    )
IMAGE_PATH = r"D:\3d-recon\datasets\ASARoomImage\d03.jpg"
TEXT_PROMPT = "wall."
BOX_TRESHOLD = 0.35
TEXT_TRESHOLD = 0.25

image_source, image = load_image(IMAGE_PATH)

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_TRESHOLD,
    text_threshold=TEXT_TRESHOLD
)

annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
cv2.imwrite("annotated_image.jpg", annotated_frame)

c:\Users\hci\.conda\envs\dino_docker\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hci\.conda\envs\dino_docker\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\hci\.conda\envs\dino_docker\lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased


c:\Users\hci\.conda\envs\dino_docker\lib\site-packages\groundingdino\util\inference.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_checkp

True

In [20]:
print(f"Boxes: {boxes}")
print(f"Logits: {logits}")
print(f"Phrases: {phrases}")

Boxes: tensor([[0.1659, 0.4498, 0.3302, 0.8986],
        [0.5187, 0.4059, 0.3832, 0.3321],
        [0.8534, 0.3232, 0.2913, 0.6198]])
Logits: tensor([0.7092, 0.5787, 0.6139])
Phrases: ['wall', 'wall', 'wall']


In [26]:
from torchvision.ops import box_convert    
    
h, w, _ = image_source.shape
boxes = boxes * torch.Tensor([w, h, w, h])
xyxy = box_convert(boxes=boxes, in_fmt="cxcywh", out_fmt="xyxy").numpy()

print(xyxy)

[[3.16850000e+03 8.20375000e+02 1.22032375e+06 1.47303312e+06]
 [1.20596825e+06 3.92939875e+05 2.61850225e+06 9.37122125e+05]
 [2.60896100e+06 2.18216875e+04 3.68297300e+06 1.03729444e+06]]


In [22]:
image.size()
img_h = image.size()[1]
img_w = image.size()[2]

In [24]:
def cxcywh_norm_to_xyxy_abs(boxes, img_w, img_h):
    """
    Convert normalized center-based boxes [cx, cy, w, h]
    to absolute pixel [x_min, y_min, x_max, y_max].

    Args:
        boxes (Tensor): shape (N, 4) in normalized [cx, cy, w, h] format.
        img_w (int): image width in pixels.
        img_h (int): image height in pixels.

    Returns:
        Tensor: shape (N, 4) in absolute [x_min, y_min, x_max, y_max] format.
    """
    boxes_abs = boxes.clone()
    boxes_abs[:, [0, 2]] *= img_w  # cx, w → pixels
    boxes_abs[:, [1, 3]] *= img_h  # cy, h → pixels

    xyxy = torch.zeros_like(boxes_abs)
    xyxy[:, 0] = boxes_abs[:, 0] - boxes_abs[:, 2] / 2  # x_min
    xyxy[:, 1] = boxes_abs[:, 1] - boxes_abs[:, 3] / 2  # y_min
    xyxy[:, 2] = boxes_abs[:, 0] + boxes_abs[:, 2] / 2  # x_max
    xyxy[:, 3] = boxes_abs[:, 1] + boxes_abs[:, 3] / 2  # y_max

    return xyxy

img_w = 1200
img_h = 800

xyxy = cxcywh_norm_to_xyxy_abs(boxes, img_w, img_h)
print(xyxy)


tensor([[1.0314e+00, 4.0051e-01, 3.9724e+02, 7.1925e+02],
        [3.9257e+02, 1.9187e+02, 8.5238e+02, 4.5758e+02],
        [8.4927e+02, 1.0655e+01, 1.1989e+03, 5.0649e+02]])


849 10 1198 506
